# Practical 4: Anomaly Detection using an Autoencoder

**Aim:** Use an autoencoder to detect anomalies (abnormal heartbeats) in ECG signals.

**Dataset:** ECG5000 (`datasets/ecg.csv`). 4,998 heartbeats; each row has 140 numbers (the ECG signal of one beat) and a final label column:
- `1` = normal heartbeat
- `0` = abnormal heartbeat (anomaly)

**Core idea:**
1. Train the autoencoder **only on normal heartbeats**. It becomes very good at rebuilding normal beats.
2. When it sees an abnormal beat it has never learned, the rebuilt signal is poor, so the **reconstruction error** is high.
3. Pick a threshold on the reconstruction error: above the threshold means anomaly.

No anomaly labels are used for training. Labels are used only at the end to measure how well the detector works.

**Steps (as per syllabus):**
- a. Import required libraries
- b. Upload / access the dataset
- c. Encoder converts it into latent representation
- d. Decoder network converts it back to the original input
- e. Compile the models with Optimizer, Loss, and Evaluation Metrics

## Step a: Import required libraries

In [ ]:
# ---------------------------------------------------------------
# Common setup: imports, reproducibility, and offline data folder
# ---------------------------------------------------------------
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # hide TensorFlow INFO/WARNING logs so outputs stay readable

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

# Fix the random seeds (Python, NumPy, TensorFlow) so every run gives similar numbers.
# This makes it easy to compare results between students.
keras.utils.set_random_seed(42)

# All data is read from the local "datasets" folder that sits next to this notebook.
# Nothing is downloaded from the internet.
DATA_DIR = Path("datasets")
MODEL_DIR = Path("models")
assert DATA_DIR.exists(), "datasets folder not found - open this notebook from the practicals folder"

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from sklearn.decomposition import PCA

## Step b: Access the dataset

In [ ]:
# The file has no header row, so header=None
df = pd.read_csv(DATA_DIR / "ecg.csv", header=None)
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
# Split into signal values (first 140 columns) and label (last column)
raw = df.values
X = raw[:, :-1].astype("float32")
labels = raw[:, -1].astype(int)

print("X shape:", X.shape, "-> 140 time steps per heartbeat")
print("Label counts: normal (1) =", (labels == 1).sum(), "| abnormal (0) =", (labels == 0).sum())
print("Signal value range:", X.min(), "to", X.max())
print("Any missing values:", df.isna().any().any())

In [ ]:
# Plot a few normal and abnormal heartbeats, and the average of each group
normal_X, abnormal_X = X[labels == 1], X[labels == 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(5):
    axes[0].plot(normal_X[i], color="tab:blue", alpha=0.6)
    axes[1].plot(abnormal_X[i], color="tab:red", alpha=0.6)
axes[0].set_title("Normal heartbeats (5 examples)")
axes[1].set_title("Abnormal heartbeats (5 examples)")
axes[2].plot(normal_X.mean(0), color="tab:blue", label="normal mean")
axes[2].plot(abnormal_X.mean(0), color="tab:red", label="abnormal mean")
axes[2].set_title("Average shape"); axes[2].legend()
for a in axes: a.set_xlabel("time step")
plt.tight_layout(); plt.show()

### Train/test split and scaling

1. Split 80/20 **before** anything else, with `stratify` so both parts keep the same normal/abnormal ratio.
2. **Min-max scale** to [0, 1] using the minimum and maximum of the **training** data only. Using test data to compute the scaling would leak information from the test set.
3. Keep only the **normal** beats of the training split for training the autoencoder.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=21, stratify=labels)

train_min, train_max = X_train.min(), X_train.max()
X_train = (X_train - train_min) / (train_max - train_min)
X_test  = (X_test  - train_min) / (train_max - train_min)

normal_train = X_train[y_train == 1]      # used for training
normal_test  = X_test[y_test == 1]
anomaly_test = X_test[y_test == 0]

print("scaled train range:", X_train.min().round(3), "to", X_train.max().round(3))
print("normal beats for training  :", normal_train.shape)
print("test set                   :", X_test.shape,
      f"({len(normal_test)} normal, {len(anomaly_test)} abnormal)")

## Step c: Encoder converts the input into a latent representation

The encoder squeezes 140 numbers down to 8 numbers (the **latent vector**, also called the bottleneck or code):

`140 -> Dense 32 -> Dense 16 -> Dense 8 (latent)`

Because 8 is much smaller than 140, the network cannot simply copy the input. It is forced to learn the most important patterns of a normal heartbeat.

In [ ]:
INPUT_DIM = X_train.shape[1]   # 140
LATENT_DIM = 8

encoder = keras.Sequential([
    keras.Input(shape=(INPUT_DIM,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(LATENT_DIM, activation="relu", name="latent"),
], name="encoder")
encoder.summary()

## Step d: Decoder converts the latent representation back to the original input

The decoder mirrors the encoder: `8 -> Dense 16 -> Dense 32 -> Dense 140`.

The final layer uses **sigmoid** so the output is in [0, 1], the same range as the scaled input.

In [ ]:
decoder = keras.Sequential([
    keras.Input(shape=(LATENT_DIM,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(INPUT_DIM, activation="sigmoid"),
], name="decoder")
decoder.summary()

In [ ]:
# The full autoencoder = encoder followed by decoder.
# Keeping them as separate models lets us look at the latent vector on its own later.
inputs = keras.Input(shape=(INPUT_DIM,))
autoencoder = keras.Model(inputs, decoder(encoder(inputs)), name="autoencoder")
autoencoder.summary()

In [ ]:
# Inspect the data flow for one heartbeat BEFORE training
sample = normal_test[:1]
z = encoder.predict(sample, verbose=0)
recon = decoder.predict(z, verbose=0)
print("input shape :", sample.shape)
print("latent shape:", z.shape, "->", np.round(z[0], 3))
print("output shape:", recon.shape)
print("reconstruction error (MAE) before training:", float(np.mean(np.abs(sample - recon))))

## Step e: Compile the model with Optimizer, Loss and Evaluation Metrics

- **Optimizer:** Adam, learning rate 0.001.
- **Loss:** Mean Absolute Error (MAE) between input and reconstruction. MAE is less dominated by a few large errors than MSE, which suits noisy signals.
- **Metric:** Mean Squared Error (MSE), reported for comparison.
- **Input = target:** we call `fit(normal_train, normal_train)`. The autoencoder learns to output its own input.

In [ ]:
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                    loss="mae",
                    metrics=["mse"])

history = autoencoder.fit(
    normal_train, normal_train,                  # input and target are the same
    epochs=40, batch_size=64,
    validation_data=(normal_test, normal_test),  # validate on unseen NORMAL beats
    shuffle=True, verbose=0)

h = history.history
print("final train loss (MAE):", round(h["loss"][-1], 4), "| final val loss (MAE):", round(h["val_loss"][-1], 4))

plt.plot(h["loss"], label="train loss (MAE)")
plt.plot(h["val_loss"], label="validation loss (MAE)")
plt.xlabel("epoch"); plt.ylabel("MAE"); plt.title("Autoencoder training"); plt.legend(); plt.show()

## Detecting anomalies with the trained autoencoder

### Inspect reconstructions
The red shaded area is the error between the input and the reconstruction. It is small for normal beats and large for abnormal beats.

In [ ]:
def plot_reconstruction(signal, title, ax):
    rec = autoencoder.predict(signal[np.newaxis], verbose=0)[0]
    err = np.mean(np.abs(signal - rec))
    ax.plot(signal, "b", label="input")
    ax.plot(rec, "r", label="reconstruction")
    ax.fill_between(np.arange(INPUT_DIM), signal, rec, color="lightcoral", alpha=0.6)
    ax.set_title(f"{title}  |  error = {err:.4f}")
    ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
plot_reconstruction(normal_test[0], "Normal", axes[0, 0])
plot_reconstruction(normal_test[1], "Normal", axes[0, 1])
plot_reconstruction(anomaly_test[0], "Abnormal", axes[1, 0])
plot_reconstruction(anomaly_test[1], "Abnormal", axes[1, 1])
plt.tight_layout(); plt.show()

In [ ]:
# Latent representation of the same beats: 140 numbers compressed to 8
print("Latent vector of a normal beat  :", np.round(encoder.predict(normal_test[:1], verbose=0)[0], 3))
print("Latent vector of an abnormal beat:", np.round(encoder.predict(anomaly_test[:1], verbose=0)[0], 3))

### Choose the threshold

Compute the reconstruction error of every **normal training** beat. A common simple rule:

`threshold = mean(train error) + 1 * std(train error)`

Anything with a higher error is flagged as an anomaly.

In [ ]:
def reconstruction_error(data):
    rec = autoencoder.predict(data, verbose=0)
    return np.mean(np.abs(data - rec), axis=1)     # one MAE value per heartbeat

train_err = reconstruction_error(normal_train)
threshold = train_err.mean() + train_err.std()
print(f"train error mean = {train_err.mean():.4f}, std = {train_err.std():.4f}")
print(f"threshold        = {threshold:.4f}")

test_err_normal = reconstruction_error(normal_test)
test_err_anom = reconstruction_error(anomaly_test)

plt.figure(figsize=(10, 4))
plt.hist(test_err_normal, bins=50, alpha=0.7, label="normal (test)")
plt.hist(test_err_anom, bins=50, alpha=0.7, label="abnormal (test)")
plt.axvline(threshold, color="k", linestyle="--", label=f"threshold {threshold:.3f}")
plt.xlabel("reconstruction error (MAE)"); plt.ylabel("number of beats"); plt.legend()
plt.title("The two groups separate clearly by reconstruction error"); plt.show()

### Evaluate the detector

The positive class here is **anomaly**. So:
- **Precision** = of the beats we flagged as anomalies, how many really were anomalies.
- **Recall** = of all real anomalies, how many we caught.

In [ ]:
test_err = reconstruction_error(X_test)
y_true_anom = (y_test == 0).astype(int)       # 1 = anomaly, 0 = normal
y_pred_anom = (test_err > threshold).astype(int)

print(f"Accuracy : {accuracy_score(y_true_anom, y_pred_anom):.4f}")
print(f"Precision: {precision_score(y_true_anom, y_pred_anom):.4f}")
print(f"Recall   : {recall_score(y_true_anom, y_pred_anom):.4f}")
print(f"F1 score : {f1_score(y_true_anom, y_pred_anom):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_true_anom, test_err):.4f}  (threshold independent)")

cm = confusion_matrix(y_true_anom, y_pred_anom)
ConfusionMatrixDisplay(cm, display_labels=["normal", "anomaly"]).plot(colorbar=False)
plt.title("Confusion matrix (test)"); plt.show()

In [ ]:
# Effect of the threshold: a lower threshold catches more anomalies (higher recall)
# but raises more false alarms (lower precision). Choose based on the cost of each mistake.
print(f"{'k':>4} {'threshold':>10} {'precision':>10} {'recall':>8} {'f1':>6}")
for k in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]:
    t = train_err.mean() + k * train_err.std()
    p = (test_err > t).astype(int)
    print(f"{k:>4} {t:>10.4f} {precision_score(y_true_anom, p, zero_division=0):>10.3f} "
          f"{recall_score(y_true_anom, p):>8.3f} {f1_score(y_true_anom, p):>6.3f}")

fpr, tpr, _ = roc_curve(y_true_anom, test_err)
plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("false positive rate"); plt.ylabel("true positive rate"); plt.title("ROC curve"); plt.show()

In [ ]:
# Visualise the 8-D latent space in 2-D with PCA.
# Normal and abnormal beats occupy different regions, even though labels were never used in training.
Z = encoder.predict(X_test, verbose=0)
Z2 = PCA(n_components=2).fit_transform(Z)
plt.figure(figsize=(6, 5))
plt.scatter(Z2[y_test == 1, 0], Z2[y_test == 1, 1], s=8, alpha=0.6, label="normal")
plt.scatter(Z2[y_test == 0, 0], Z2[y_test == 0, 1], s=8, alpha=0.6, label="abnormal")
plt.title("Latent space (PCA to 2-D)"); plt.legend(); plt.show()

## Try it yourself (optional)

1. Change `LATENT_DIM` to 2 and to 32. Too small loses detail even for normal beats; too large lets the network copy everything, including anomalies. Both hurt detection.
2. Change the loss to `"mse"` and compare the ROC AUC.
3. Train on **all** beats (normal + abnormal) instead of only normal ones. Detection gets worse, because the autoencoder also learns to rebuild anomalies.

## Conclusion

An autoencoder trained only on normal heartbeats rebuilds normal beats with low error and abnormal beats with high error. A threshold on the reconstruction error turns it into an anomaly detector that reaches over 90% accuracy on ECG5000 without using any anomaly labels during training.